In [2]:
import medmnist
from medmnist import OCTMNIST
from medmnist import INFO
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from torchvision import models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from utils.dataset import Dataset 
from dotenv import load_dotenv
import os

ImportError: cannot import name 'ASSETS_DIR' from partially initialized module 'utils' (most likely due to a circular import) (c:\Users\feder\Desktop\Magistrale\Biometria\Progetto_Quantum\ProgettoQuantumBiometria\src\utils\__init__.py)

In [ ]:
data_flag = 'octmnist'

download = True

load_dotenv()

device = os.getenv('DEVICE', 'cpu')

print(f"Using device: {device}")


info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

In [ ]:
backbone = resnet18(weights=models.ResNet18_Weights.DEFAULT)
backbone.fc = torch.nn.Identity()
backbone.eval()

weights = models.ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
print(preprocess.mean)


# Definisci le trasformazioni con la conversione RGB in cima
preprocess = transforms.Compose([
    # 1. Converte l'immagine PIL da scala di grigio a RGB (3 canali identici)
    transforms.Lambda(lambda img: img.convert('RGB')), 
    
    # 2. Converte in tensore PyTorch
    transforms.ToTensor(),
    
    # 3. Ora la normalizzazione funzionerà perfettamente perché il tensore ha 3 canali
    transforms.Normalize(preprocess.mean, preprocess.std)
])


In [ ]:
data_flag = "octmnist"
download = True

# Load MedMNIST datasets directly - NO concatenation!
train_dataset = DataClass(
     split="train", transform=preprocess, download=download, size=224
)
val_dataset = DataClass(
     split="val", transform=preprocess, download=download, size=224
)
test_dataset = DataClass(
     split="test", transform=preprocess, download=download, size=224
)     

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Val Dataset: {len(val_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")
data_toprocess = {
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset
}

In [ ]:
# Helper function to extract features from a dataset without loading everything to memory
def extract_features_from_dataset(dataset, backbone, batch_size=32, device='cpu'):
    """
    Extract features and labels from a dataset in batches.
    Returns numpy arrays of features and labels.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features_list = []
    labels_list = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            # Extract features using backbone
            pooled = backbone(imgs)
            flat = pooled.view(pooled.size(0), -1)  # (batch, 512)
            
            # Move back to CPU for storage
            features_list.append(flat.cpu().numpy())
            labels_list.append(labels.numpy())
    
    # Concatenate all batches (handles variable batch sizes, including incomplete final batch)
    all_features = np.concatenate(features_list, axis=0)  # Shape: (num_samples, 512)
    all_labels = np.concatenate(labels_list, axis=0).flatten()  # Shape: (num_samples,)
    
    return all_features, all_labels


# Set device (GPU if available)
print(f"Using device: {device}")

backbone_device = backbone.to(device)

dataframe = pd.DataFrame()


for dataset_name, dataset in data_toprocess.items():
    print(f"\n{'='*60}")
    print(f"Processing Dataset: {dataset_name}")
    print(f"{'='*60}")
    
    features, labels = extract_features_from_dataset(
        dataset, backbone_device, batch_size=32, device=device
    )
    
    print(f"{dataset_name.capitalize()} Features: {features.shape}, Labels: {labels.shape}")
    dataset = pd.DataFrame(features)
    dataset['label'] = labels
    dataframe = pd.concat([dataframe, dataset], ignore_index=True)
       


In [ ]:
del train_dataset, val_dataset, test_dataset  # Free memory after processing datasets

In [ ]:
from pathlib import Path
# Extract features for each dataset split
seeds = [11, 17, 29]
for seed in seeds:
    print(f"\n{'='*60}")
    print(f"Processing Seed: {seed}")
    print(f"{'='*60}")
    
    train, val_test = train_test_split(dataframe,
                                        test_size=0.2, random_state=seed, stratify=dataframe['label'])
    val, test = train_test_split(val_test,
                                  test_size=0.5, random_state=seed, stratify=val_test['label'])
    


    root = Path.cwd()
    print(root)  # controlla la cartella di lavoro
    path = root / "assets" / f"seeds_{seed}"
     # Save each split separately to CSV
    print(f"\n--- Saving Features to CSV for Seed: {seed} ---")
    
    os.makedirs(path, exist_ok=True)
    train.to_csv(f"{path}/train_{seed}.csv", index=False)
    val.to_csv(f"{path}/val_{seed}.csv", index=False)
    test.to_csv(f"{path}/test_{seed}.csv", index=False)